In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm
import time


def InverseF(y: float, theta: float) -> float:
    """
    обратная функция для генерации выборки из распределения Парето.
    """
    return (1 - y) ** (1 / (1 - theta))

In [2]:
# выборка и Асимптотические доверительные интервалы

N = 100
theta_true = 3.0
beta = 0.95
z_gamma = norm.ppf((1 + beta) / 2)

med_true = 2 ** (1 / (theta_true - 1))

random_numbers = np.random.random(size=N)
sample = np.array([InverseF(y, theta_true) for y in random_numbers])

# Точечные оценки (О.М.П.)
ln_mean = np.mean([np.log(x) for x in sample])
theta_hat = 1 + 1 / ln_mean
med_hat = 2 ** (1 / (theta_hat - 1))

# Асимптотические доверительные интервалы
# Для параметра theta:
margin_theta = z_gamma / (np.sqrt(N) * ln_mean)
ci_theta_asymp = (theta_hat - margin_theta, theta_hat + margin_theta)

# Для медианы:
margin_med = (z_gamma * np.log(2) * med_hat) / (np.sqrt(N) * (theta_hat - 1))
ci_med_asymp = (med_hat - margin_med, med_hat + margin_med)

print(f"Истинное theta: {theta_true:.4f} | Оценка theta_hat: {theta_hat:.4f}")
print(f"Истинная медиана: {med_true:.4f} | Оценка med_hat: {med_hat:.4f}")
print("-" * 50)
print(f"Асимпт. ДИ для theta:   [{ci_theta_asymp[0]:.4f}, {ci_theta_asymp[1]:.4f}]")
print(f"Асимпт. ДИ для медианы: [{ci_med_asymp[0]:.4f}, {ci_med_asymp[1]:.4f}]")

Истинное theta: 3.0000 | Оценка theta_hat: 2.9365
Истинная медиана: 1.4142 | Оценка med_hat: 1.4304
--------------------------------------------------
Асимпт. ДИ для theta:   [2.5570, 3.3160]
Асимпт. ДИ для медианы: [1.3300, 1.5307]


In [3]:
# Параметрический бутстрап 

B = 50_000 # Количество итераций
delta_theta_param = []
delta_med_param = []

print("Считаем параметрический бутстрап...")
start_time = time.time()

for _ in range(B):
    U_boot = np.random.random(size=N)
    boot_sample = [InverseF(y, theta_hat) for y in U_boot]
    
    boot_ln_mean = np.mean([np.log(x) for x in boot_sample])
    boot_theta = 1 + 1 / boot_ln_mean
    boot_med = 2 ** (1 / (boot_theta - 1))
    
    delta_theta_param.append(boot_theta - theta_hat)
    delta_med_param.append(boot_med - med_hat)

delta_theta_param.sort()
delta_med_param.sort()

lower_idx = int((1 - beta) * B / 2)
upper_idx = int((1 + beta) * B / 2)

ci_theta_boot_param = (theta_hat - delta_theta_param[upper_idx], 
                       theta_hat - delta_theta_param[lower_idx])

ci_med_boot_param = (med_hat - delta_med_param[upper_idx], 
                     med_hat - delta_med_param[lower_idx])

Считаем параметрический бутстрап...


In [4]:
# Непараметрический бутстрап

delta_theta_nonparam = []
delta_med_nonparam = []
B = 1000

print("Считаем непараметрический бутстрап...")
start_time = time.time()

for _ in range(B):
    boot_sample = np.random.choice(sample, size=N, replace=True)
    
    boot_ln_mean = np.mean([np.log(x) for x in boot_sample])
    boot_theta = 1 + 1 / boot_ln_mean
    boot_med = 2 ** (1 / (boot_theta - 1))
    
    delta_theta_nonparam.append(boot_theta - theta_hat)
    delta_med_nonparam.append(boot_med - med_hat)

delta_theta_nonparam.sort()
delta_med_nonparam.sort()

lower_idx = int((1 - beta) * B / 2)
upper_idx = int((1 + beta) * B / 2) 

ci_theta_boot_nonparam = (theta_hat - delta_theta_nonparam[upper_idx], 
                          theta_hat - delta_theta_nonparam[lower_idx])

ci_med_boot_nonparam = (med_hat - delta_med_nonparam[upper_idx], 
                        med_hat - delta_med_nonparam[lower_idx])

Считаем непараметрический бутстрап...


In [5]:
# Сравнение интервалов 

df_theta = pd.DataFrame({
    'Метод': ['Асимптотический (ОМП)', 'Парам. Бутстрап', 'Непарам. Бутстрап'],
    'Левая граница': [ci_theta_asymp[0], ci_theta_boot_param[0], ci_theta_boot_nonparam[0]],
    'Правая граница': [ci_theta_asymp[1], ci_theta_boot_param[1], ci_theta_boot_nonparam[1]]
})
df_theta['Длина'] = df_theta['Правая граница'] - df_theta['Левая граница']
df_theta['Накрыл?'] = (df_theta['Левая граница'] <= theta_true) & (theta_true <= df_theta['Правая граница'])

df_med = pd.DataFrame({
    'Метод': ['Асимптотический', 'Парам. Бутстрап', 'Непарам. Бутстрап'],
    'Левая граница': [ci_med_asymp[0], ci_med_boot_param[0], ci_med_boot_nonparam[0]],
    'Правая граница': [ci_med_asymp[1], ci_med_boot_param[1], ci_med_boot_nonparam[1]]
})
df_med['Длина'] = df_med['Правая граница'] - df_med['Левая граница']
df_med['Накрыл?'] = (df_med['Левая граница'] <= med_true) & (med_true <= df_med['Правая граница'])

print(f"ДЛЯ ПАРАМЕТРА THETA (Истина: {theta_true})")
display(df_theta.round(4))
print("\n")
print(f"АНАЛИЗ ДЛЯ МЕДИАНЫ (Истина: {med_true:.4f})")
display(df_med.round(4))

ДЛЯ ПАРАМЕТРА THETA (Истина: 3.0)


,Метод,Левая граница,Правая граница,Длина,Накрыл?
0,Асимптотический (ОМП),2.5570,3.3160,0.7591,True
1,Парам. Бутстрап,2.4958,3.2683,0.7724,True
2,Непарам. Бутстрап,2.4406,3.3051,0.8644,True




АНАЛИЗ ДЛЯ МЕДИАНЫ (Истина: 1.4142)


,Метод,Левая граница,Правая граница,Длина,Накрыл?
0,Асимптотический,1.3300,1.5307,0.2007,True
1,Парам. Бутстрап,1.3205,1.5222,0.2017,True
2,Непарам. Бутстрап,1.3043,1.5308,0.2264,True
